Iteration of the trade matrix 

In [ ]:
import pandas as pd
import numpy as np
import os

# The trade matrix for each product needs to be iterated; here we take one as an example.
# Read Excel data
# A sheet: 32 rows and 34 columns (including interprovincial transfers, exports, export ratio) Production statistics
A_path = r'E:\Trade matrix.xlsx'  
A = pd.read_excel(A_path, header=None)

# Consumption data: Province + Values
consumption_path = r'E:\Consumption.xlsx'  
consumption_df = pd.read_excel(consumption_path)
consumption_df.columns = ['province', 'consumption']
consumption_df.set_index('province', inplace=True)  # Correctly use inplace

# Actual production data: Province + Values
actual_production_path = r'E:\Statistics_production.xlsx'  
production_df = pd.read_excel(actual_production_path)
production_df.columns = ['province', 'production']
production_df.set_index('province', inplace=True)   # Correctly use inplace

# 2. Align consumption and production order
def clean_name(name):
    return name.strip().replace("Province", "").replace("City", "").replace("Autonomous region", "").replace("Zhuang ethnic group", "").replace("Hui people", "").replace("Uighur", "")

# Clean all province names
provinces_raw = A.iloc[1:, 0].values
provinces_cleaned = pd.Series(provinces_raw).map(clean_name).values

consumption_df.index = consumption_df.index.map(clean_name)
production_df.index = production_df.index.map(clean_name)

# Print verification
print("Cleaned provinces in A:", provinces_cleaned.tolist())
print("Consumption data index:", list(consumption_df.index))
print("Production data index:", list(production_df.index))

# Get data aligned with A's order
consumption = consumption_df.loc[provinces_cleaned, 'consumption']
actual_production = production_df.loc[provinces_cleaned, 'production']
# -------------------------------
# 3. Define iteration function
def iterate_trade_matrix(A, consumption, actual_production, tolerance=0.05
                         , max_iter=20000, verbose=False):
    provinces = A.iloc[1:, 0].values
    n = len(provinces)

    transfer_matrix = A.iloc[1:n+1, 1:n+1].astype(float).values

    export_vector = A.iloc[1:n+1, 32].astype(float).values
    export_ratio = A.iloc[1:n+1, 33].astype(float).values
    for iteration in range(max_iter):
        column_sums = transfer_matrix.sum(axis=0)    # Sum of each column, gives total input for each consuming province
        column_sums[column_sums == 0] = 1  # Avoid division by 0
        proportion_matrix = transfer_matrix / column_sums  # Proportion of consumption coming from other provinces
        consumption_matrix = proportion_matrix * consumption.values.reshape(1, -1)  # Distribute consumption data according to proportion, resulting in a new interprovincial "consumption source matrix"
        estimated_production_noex = consumption_matrix.sum(axis=1) 
        ex = estimated_production_noex * export_ratio / (1 - export_ratio)
        estimated_production = estimated_production_noex + ex
         # Derived production for each province = consumption sent to other provinces + export volume
        diff_ratio = 1 - (actual_production.values / estimated_production)  # Calculate the difference ratio between derived and actual production
        print(diff_ratio)
        if verbose:
            max_error = np.max(np.abs(diff_ratio))
            print(f"Iteration {iteration+1}: Maximum error = {max_error:.4f}")  # If verbose=True, output the max error of each iteration for monitoring convergence

        if np.all(np.abs(diff_ratio) < tolerance):
            break  # Stop iteration when the error for all provinces is less than the tolerance

        for i in range(n):
            transfer_matrix[i, :] -= transfer_matrix[i, :] * diff_ratio[i]
            export_vector[i] -= export_vector[i] * diff_ratio[i]     # Scale the trade flows and export values of each province by the error ratio
    # Final trade matrix and recalculated export
    column_sums = transfer_matrix.sum(axis=0)
    column_sums[column_sums == 0] = 1
    proportion_matrix = transfer_matrix / column_sums
    final_consumption_matrix = proportion_matrix * consumption.values.reshape(1, -1)  # Re-generate the consumption distribution matrix (for output)

    new_export_vector = export_ratio * transfer_matrix.sum(axis=1) / (1 - export_ratio)  # Recalculate export volume using export ratio and total transfer to ensure consistency

    final_trade_matrix = pd.DataFrame(transfer_matrix, index=provinces, columns=provinces)
    final_export_series = pd.Series(new_export_vector, index=provinces, name='Export')
    final_consumption_matrix_df = pd.DataFrame(final_consumption_matrix, index=provinces, columns=provinces)  # Convert NumPy array into Pandas object for output and visualization

    # Derived production = total interprovincial transfer + export
    derived_output = transfer_matrix.sum(axis=1) + new_export_vector
    output_diff_ratio = 1 - (actual_production.values / derived_output)

    output_comparison = pd.DataFrame({
        'Province': provinces,
        'Derived Production': derived_output,
        'Actual Production': actual_production.values,
        'Difference Ratio': output_diff_ratio
    })

    return final_trade_matrix, final_export_series, final_consumption_matrix_df, output_comparison

# 4. Run the model
final_trade_matrix, final_export, final_consumption_matrix, output_comparison = iterate_trade_matrix(
    A, consumption, actual_production, tolerance=0.05, max_iter=20000, verbose=True)

# 5. Save all results
output_path = r'E:\China Nitrogen Cycle Data\Iterative Data'
os.makedirs(output_path, exist_ok=True)  
final_trade_matrix.to_excel(os.path.join(output_path, 'Trade matrix_after Iteration.xlsx'))
final_export.to_excel(os.path.join(output_path, 'Export_after Iteration.xlsx'))




Uncertainty Analysis

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import genextreme, norm
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

# ---------------------- 1. Core Parameter Definitions (Fixed Factors + Uncertainty Parameters Separation) ----------------------
# Production-side parameters: Fixed emission factors (used to calculate averages) + Uncertainty parameters (used for confidence interval simulation)
prod_params = {
    'urea': {'ef_fixed': 1.99, 'nitrogen_ratio': 0.46, 'std_dev': 0.18},  # Fixed factor: 1.99
    'ammonium carbonate': {'ef_fixed': 6.19, 'nitrogen_ratio': 1.0, 'std_dev': 0.18},   # Fixed factor: 6.19
    'diammonium phosphate': {'ef_fixed': 1.15, 'nitrogen_ratio': 0.18, 'std_dev': 0.18},  # Fixed factor: 1.15
    'compound fertilizer': {'ef_fixed': 1.71, 'nitrogen_ratio': 0.15, 'std_dev': 0.18}  # Fixed factor: 1.71
}

# Application-side parameters: Fixed emission factors (used to calculate averages) + Uncertainty parameters (used for confidence interval simulation)
apply_ef_params = {
    # Fixed factors (your input values)
    'fixed': {
        'urea_EF1': 0.011,
        'ammonium_carbonate_EF1': 0.012,
        'diammonium_phosphate_EF1': 0.009,
        'compound_fertilizer_EF1': 0.008,
        'FracGASF': 0.11,
        'EF4': 0.01,
        'FracLEACH': 0.24,
        'EF5': 0.011
    },
    # Uncertainty parameters (used for confidence interval simulation)
    'uncertain': {
        'urea_EF1': {'mean': 0.011, 'lower': 0.011*0.6, 'upper': 0.011*1.7, 'dist_type': 'GEV'},
        'ammonium_carbonate_EF1': {'mean': 0.012, 'lower': 0.012*0.6, 'upper': 0.012*1.7, 'dist_type': 'GEV'},
        'diammonium_phosphate_EF1': {'mean': 0.009, 'lower': 0.009*0.6, 'upper': 0.009*1.7, 'dist_type': 'GEV'},
        'compound_fertilizer_EF1': {'mean': 0.008, 'lower': 0.008*0.6, 'upper': 0.008*1.7, 'dist_type': 'GEV'},
        'FracGASF': {'mean': 0.11, 'lower': 0.11*0.6, 'upper': 0.11*1.7, 'dist_type': 'GEV'},
        'EF4': {'mean': 0.01, 'lower': 0.01*0.8, 'upper': 0.01*1.2, 'dist_type': 'Normal'},
        'FracLEACH': {'mean': 0.24, 'lower': 0.24*0.6, 'upper': 0.24*1.7, 'dist_type': 'GEV'},
        'EF5': {'mean': 0.011, 'lower': 0.011*0.6, 'upper': 0.011*1.7, 'dist_type': 'GEV'}
    }
}

# Monte Carlo simulation parameters
n_simulations = 5000
excel_input_path = 'E:\Nitrogen fertilizer.xlsx'
excel_output_path = 'E:\Uncertainty_emissions.xlsx'

# ---------------------- 2. Distribution Fitting Functions (Only for Confidence Interval Simulation) ----------------------
def fit_distribution(params):
    if params['dist_type'] == 'GEV':
        ci_samples = np.linspace(params['lower'], params['upper'], 500)
        shape, loc, scale = genextreme.fit(ci_samples)
        return lambda size: genextreme.rvs(shape, loc=loc, scale=scale, size=size)
    elif params['dist_type'] == 'Normal':
        std = (params['upper'] - params['lower']) / (2 * 1.96)
        return lambda size: norm.rvs(loc=params['mean'], scale=std, size=size)

# Pre-fit emission factor generators (only for confidence interval simulation)
apply_dist_generators = {key: fit_distribution(val) for key, val in apply_ef_params['uncertain'].items()}

# Production-side emission factor generators (only for confidence interval simulation: 18% standard deviation normal distribution)
prod_ef_generators = {}
for fert, params in prod_params.items():
    ef_mean = params['ef_fixed']  # Centered around the fixed factor
    ef_std = ef_mean * params['std_dev']  # 18% standard deviation
    prod_ef_generators[fert] = lambda size, m=ef_mean, s=ef_std: norm.rvs(loc=m, scale=s, size=size)

# ---------------------- 3. Data Reading Functions ----------------------
def read_fertilizer_data(sheet_name):
    df = pd.read_excel(excel_input_path, sheet_name=sheet_name)
    df = df.iloc[0:33].copy()  # Read rows 1-33 (including provinces + national data)
    df.columns = ['Province', 'Urea Production_10kton', 'Ammonium Carbonate Production_10kton', 'Diammonium Phosphate Production_10kton', 'Compound Fertilizer Production_10kton',
                  'Urea Application_ton', 'Ammonium Carbonate Application_ton', 'Diammonium Phosphate Application_ton', 'Compound Fertilizer Application_ton']
    df = df.fillna(0)
    df['Province'] = df['Province'].str.strip()  # Remove spaces to avoid mismatches
    return df

# Read data for three years
df_2021 = read_fertilizer_data('2021')
df_2030 = read_fertilizer_data('2030')
df_2050 = read_fertilizer_data('2050')

# ---------------------- 4. Core Emission Calculation Function (Key Modification: Separate Fixed Values and Uncertainty) ----------------------
def calculate_emission_uncertainty(df, year):
    result_list = []
    fixed_params = apply_ef_params['fixed']  # Your fixed emission factors
    
    for _, row in df.iterrows():
        province = row['Province']
        prod_data = {
            'urea': row['Urea Production_10kton'], 'ammonium carbonate': row['Ammonium Carbonate Production_10kton'],
            'diammonium phosphate': row['Diammonium Phosphate Production_10kton'], 'compound fertilizer': row['Compound Fertilizer Production_10kton']
        }
        apply_data = {
            'urea': row['Urea Application_ton'], 'ammonium carbonate': row['Ammonium Carbonate Application_ton'],
            'diammonium phosphate': row['Diammonium Phosphate Application_ton'], 'compound fertilizer': row['Compound Fertilizer Application_ton']
        }
        
        # ---------------------- 1. Fixed Value Calculation (Average Value: Use Fixed Factors, No Uncertainty) ----------------------
        # Production-side fixed value (matches your formula exactly)
        prod_fixed = 0
        for fert, prod_10kton in prod_data.items():
            if prod_10kton == 0:
                continue
            params = prod_params[fert]
            prod_fixed += prod_10kton * 10000 / params['nitrogen_ratio'] * params['ef_fixed']
        
        # Application-side fixed value (use your fixed factors)
        direct_n2o_fixed = (
            apply_data['urea'] * fixed_params['urea_EF1'] * (44*273/28) +
            apply_data['ammonium carbonate'] * fixed_params['ammonium_carbonate_EF1'] * (44*273/28) +
            apply_data['diammonium phosphate'] * fixed_params['diammonium_phosphate_EF1'] * (44*273/28) +
            apply_data['compound fertilizer'] * fixed_params['compound_fertilizer_EF1'] * (44*273/28)
        )
        total_apply = sum(apply_data.values())
        nh3_volatilization_fixed = total_apply * fixed_params['FracGASF'] * fixed_params['EF4'] * (44*273/28)
        nitrate_leaching_fixed = total_apply * fixed_params['FracLEACH'] * fixed_params['EF5'] * (44*273/28)
        urea_carbamide_decomp_fixed = apply_data['urea']*1.57 + apply_data['ammonium carbonate']*3.14
        limestone_fixed = (apply_data['urea']*1.8 + apply_data['ammonium carbonate']*1.81 +
                          apply_data['diammonium phosphate']*5.15 + apply_data['compound fertilizer']*1.79) * (0.12*44/12)
        apply_fixed = direct_n2o_fixed + nh3_volatilization_fixed + nitrate_leaching_fixed + urea_carbamide_decomp_fixed + limestone_fixed
        total_fixed = prod_fixed + apply_fixed
        
        # ---------------------- 2. Uncertainty Simulation (Only for Confidence Interval Calculation) ----------------------
        # Production-side uncertainty simulation
        prod_emission_sim = np.zeros(n_simulations)
        for fert, prod_10kton in prod_data.items():
            if prod_10kton == 0:
                continue
            params = prod_params[fert]
            ef_rand = prod_ef_generators[fert](size=n_simulations)
            prod_emission_sim += prod_10kton * 10000 / params['nitrogen_ratio'] * ef_rand
        
        # Application-side uncertainty simulation
        ef1_urea_rand = apply_dist_generators['urea_EF1'](size=n_simulations)
        ef1_ammonium_carbonate_rand = apply_dist_generators['ammonium_carbonate_EF1'](size=n_simulations)
        ef1_diammonium_phosphate_rand = apply_dist_generators['diammonium_phosphate_EF1'](size=n_simulations)
        ef1_compound_fertilizer_rand = apply_dist_generators['compound_fertilizer_EF1'](size=n_simulations)
        frac_gasf_rand = apply_dist_generators['FracGASF'](size=n_simulations)
        ef4_rand = apply_dist_generators['EF4'](size=n_simulations)
        frac_leach_rand = apply_dist_generators['FracLEACH'](size=n_simulations)
        ef5_rand = apply_dist_generators['EF5'](size=n_simulations)
        
        direct_n2o_rand = (
            apply_data['urea'] * ef1_urea_rand * (44*273/28) +
            apply_data['ammonium carbonate'] * ef1_ammonium_carbonate_rand * (44*273/28) +
            apply_data['diammonium phosphate'] * ef1_diammonium_phosphate_rand * (44*273/28) +
            apply_data['compound fertilizer'] * ef1_compound_fertilizer_rand * (44*273/28)
        )
        nh3_volatilization_rand = total_apply * frac_gasf_rand * ef4_rand * (44*273/28)
        nitrate_leaching_rand = total_apply * frac_leach_rand * ef5_rand * (44*273/28)
        urea_carbamide_decomp_rand = np.full(n_simulations, urea_carbamide_decomp_fixed)  # No uncertainty
        limestone_rand = np.full(n_simulations, limestone_fixed)  # No uncertainty
        
        apply_emission_sim = direct_n2o_rand + nh3_volatilization_rand + nitrate_leaching_rand + urea_carbamide_decomp_rand + limestone_rand
        total_emission_sim = prod_emission_sim + apply_emission_sim
        
        # ---------------------- 3. Statistical Confidence Interval (Extract from Simulation Results) ----------------------
        # Production-side confidence interval
        prod_ci_lower = np.percentile(prod_emission_sim, 2.5)
        prod_ci_upper = np.percentile(prod_emission_sim, 97.5)
        
        # Application-side confidence interval
        apply_ci_lower = np.percentile(apply_emission_sim, 2.5)
        apply_ci_upper = np.percentile(apply_emission_sim, 97.5)
        
        # Total emissions confidence interval
        total_ci_lower = np.percentile(total_emission_sim, 2.5)
        total_ci_upper = np.percentile(total_emission_sim, 97.5)
        
        # Print results (fixed values + confidence intervals)
        print(f"\n===== {year} - {province} =====")
        print(f"Production: Fixed value {round(prod_fixed, 2)} tons CO2 (95% CI: [{round(prod_ci_lower, 2)}, {round(prod_ci_upper, 2)}])")
        print(f"Application: Fixed value {round(apply_fixed, 2)} tons CO2 (95% CI: [{round(apply_ci_lower, 2)}, {round(apply_ci_upper, 2)}])")
        print(f"Total: Fixed value {round(total_fixed, 2)} tons CO2 (95% CI: [{round(total_ci_lower, 2)}, {round(total_ci_upper, 2)}])")
        print("-" * 60)
        
        # Save results (fixed values as averages, confidence intervals from simulation)
        result_list.append({
            'Year': year,
            'Province': province,
            # Production side (fixed values + confidence intervals)
            'Production Emission_tons CO2 (Fixed)': round(prod_fixed, 2),
            'Production 95% CI Lower_tons CO2': round(prod_ci_lower, 2),
            'Production 95% CI Upper_tons CO2': round(prod_ci_upper, 2),
            # Application side (fixed values + confidence intervals)
            'Application Emission_tons CO2 (Fixed)': round(apply_fixed, 2),
            'Application 95% CI Lower_tons CO2': round(apply_ci_lower, 2),
            'Application 95% CI Upper_tons CO2': round(apply_ci_upper, 2),
            # Total emissions (fixed values + confidence intervals)
            'Total Emission_tons CO2 (Fixed)': round(total_fixed, 2),
            'Total 95% CI Lower_tons CO2': round(total_ci_lower, 2),
            'Total 95% CI Upper_tons CO2': round(total_ci_upper, 2),
            'Total Uncertainty Range_tons CO2': round(total_ci_upper - total_ci_lower, 2)
        })
    
    return pd.DataFrame(result_list)

# ---------------------- 5. Execute Calculations and Save Results ----------------------
result_2021 = calculate_emission_uncertainty(df_2021, 2021)
result_2030 = calculate_emission_uncertainty(df_2030, 2030)
result_2050 = calculate_emission_uncertainty(df_2050, 2050)

# Save to Excel
with pd.ExcelWriter(excel_output_path, engine='openpyxl') as writer:
    result_2021.to_excel(writer, sheet_name='2021 Results', index=False)
    result_2030.to_excel(writer, sheet_name='2030 Results', index=False)
    result_2050.to_excel(writer, sheet_name='2050 Results', index=False)
    result_all = pd.concat([result_2021, result_2030, result_2050], ignore_index=True)
    result_all.to_excel(writer, sheet_name='Three-Year Comparison', index=False)

# ---------------------- 6. Interval Line Chart Plotting (Key Modification: Piecewise Interpolation + Correct Display) ----------------------
def plot_interval_line_chart(province='National'):
    # Extract data for the province over three years
    df_province = result_all[result_all['Province'].str.strip() == province.strip()].sort_values('Year')
    
    # Data validation
    if len(df_province) != 3:
        print(f"\nWarning: Only {len(df_province)} years of data found for {province} (needs 2021, 2030, 2050)")
        return
    
    # Extract raw data (in tons CO2)
    years_original = df_province['Year'].values  # [2021, 2030, 2050]
    total_fixed_original = df_province['Total Emission_tons CO2 (Fixed)'].values  # Correct values in the table (e.g., 375Mt = 375000000 tons)
    total_lower_original = df_province['Total 95% CI Lower_tons CO2'].values     # In tons
    total_upper_original = df_province['Total 95% CI Upper_tons CO2'].values     # In tons
    print(total_fixed_original)
    
    # ---------------------- Key Modification: Convert Tons → Megatons (Mt) ----------------------
    total_fixed_original = total_fixed_original / 1000000  # 375000000 tons → 375 Mt
    total_lower_original = total_lower_original / 1000000
    total_upper_original = total_upper_original / 1000000
    
    # Generate continuous years from 2021 to 2050 (to ensure the line and confidence interval display continuously)
    interp_years_full = np.arange(2021, 2051)  # [2021,2022,...,2050]
    
    # Piecewise interpolation: 2021→2030 and 2030→2050 (interpolation logic remains the same, but data is now converted to Mt)
    # Interpolation from 2021 to 2030
    f_fixed1 = interp1d(years_original[:2], total_fixed_original[:2], kind='linear')
    f_lower1 = interp1d(years_original[:2], total_lower_original[:2], kind='linear')
    f_upper1 = interp1d(years_original[:2], total_upper_original[:2], kind='linear')
    interp_fixed1 = f_fixed1(interp_years_full[interp_years_full <= 2030])
    interp_lower1 = f_lower1(interp_years_full[interp_years_full <= 2030])
    interp_upper1 = f_upper1(interp_years_full[interp_years_full <= 2030])
    
    # Interpolation from 2030 to 2050
    f_fixed2 = interp1d(years_original[1:], total_fixed_original[1:], kind='linear')
    f_lower2 = interp1d(years_original[1:], total_lower_original[1:], kind='linear')
    f_upper2 = interp1d(years_original[1:], total_upper_original[1:], kind='linear')
    interp_fixed2 = f_fixed2(interp_years_full[interp_years_full > 2030])
    interp_lower2 = f_lower2(interp_years_full[interp_years_full > 2030])
    interp_upper2 = f_upper2(interp_years_full[interp_years_full > 2030])
    
    # Concatenate the full interpolation result (in Mt)
    interp_fixed_full = np.concatenate([interp_fixed1, interp_fixed2])
    interp_lower_full = np.concatenate([interp_lower1, interp_lower2])
    interp_upper_full = np.concatenate([interp_upper1, interp_upper2])
    
    # Plotting settings
    plt.rcParams['font.sans-serif'] = ['Arial']  # English font
    plt.rcParams['axes.unicode_minus'] = False    # Support for negative signs
    plt.figure(figsize=(12, 7))
    
    # Confidence interval fill (in Mt)
    plt.fill_between(interp_years_full, interp_lower_full, interp_upper_full, 
                     color='lightgray', alpha=0.6, label='95% Confidence Interval')
    
    # Fixed value line (red thick line, in Mt, consistent with table values)
    plt.plot(interp_years_full, interp_fixed_full, 'red', linewidth=2, label='BAU')
    # Legend: Adjust font, size, remove gray border

    # Original year markers (real fixed values for 2021, 2030, 2050, in Mt)
    plt.scatter(years_original, total_fixed_original, color='red', s=80, zorder=5)
    plt.scatter(years_original, total_lower_original, color='gray', s=50, zorder=5)
    plt.scatter(years_original, total_upper_original, color='gray', s=50, zorder=5)
    
    # Beautify the chart
    plt.xlabel('Year', fontsize=16, fontname='Arial')
    plt.ylabel(r'GHG emissions (Mt CO$_2$-eq)', fontsize=16, fontname='Arial')  # Correct unit
    #plt.title(f'{province} GHG Emission Uncertainty Trend (2021-2050)', fontsize=14, fontweight='bold')
    plt.legend(
        fontsize=14,
        frameon=False
        )
    
    # X-axis ticks: 2021, 2030, 2040, 2050
    plt.xticks([2021, 2030, 2040, 2050], fontsize=14, fontname='Arial' )
    plt.xlim(2020, 2051)  # Leave space on both sides for better display
    
    # Y-axis: Start from 0, auto-adjust to Mt scale (e.g., 375Mt)
    y_max = np.max([np.max(interp_upper_full), np.max(total_upper_original)])

    plt.yticks(np.arange(0, y_max * 1.1, step=50), fontsize=14, fontname='Arial')  # Set ticks by step of 50, adjust as needed
    plt.tight_layout()

    plt.savefig(f'{province}_Carbon_Emission_Trend_FixedLine.png', dpi=1200, bbox_inches='tight', pil_kwargs={'compress_level':0})
    plt.show()
    # Plot national trend (example)
print("\n===== Available Provinces/National List =====")
print(result_all['Province'].unique())
plot_interval_line_chart(province='National')

print(f"\nComputation completed! Results have been saved to: {excel_output_path}")
